# Plan Mode Data Flow

**Purpose:** Answer three research questions from the rodin v2 proposal about what data hooks receive during plan mode entry and exit, and how to locate the plan file.

| # | Question | Why it matters |
|---|----------|----------------|
| 1 | What data does PostToolUse receive from EnterPlanMode? | The template injection hook needs to know the plan file path to inject structure |
| 2 | What data does PreToolUse receive for ExitPlanMode? | The exit review hook needs to locate and read the plan file |
| 6 | How does the agent hook locate the plan file? | Without a reliable path, the review can't read the plan |

Source: `openspec/changes/rodin/proposal.md` — Outstanding research questions 1, 2, 6

Plugin: `research/plugins/full-capture/` — captures both PreToolUse and PostToolUse events to JSONL

## Background

The rodin v2 design uses two hooks:

1. **PostToolUse on EnterPlanMode** — injects a plan template into the plan file
2. **PreToolUse on ExitPlanMode** — reviews the plan against conversation before allowing exit

Both hooks need to know the **plan file path**. The plan file path appears in the system prompt injection when entering plan mode, but it's unclear whether hooks can access it from the tool input or response.

### What we know from prior research

The `post-tool-use-hook-input` notebook established the PostToolUse envelope shape:

```json
{
  "session_id": "<uuid>",
  "tool_name": "<ToolName>",
  "tool_input": { ... },
  "tool_response": { ... }
}
```

But we haven't captured EnterPlanMode or ExitPlanMode events specifically. These tools have unusual shapes:

- **EnterPlanMode**: No required parameters (empty `tool_input`?)
- **ExitPlanMode**: Only optional parameters (`allowedPrompts`, `pushToRemote`, etc.)

The question is: does `tool_response` for EnterPlanMode contain the plan file path?

In [ ]:
# Setup — imports, plugin paths, capture file locations
import json
from pathlib import Path
from lib import run_claude

PLUGIN_DIR = Path("plugins/full-capture").resolve()
PRE_CAPTURE = Path("/tmp/claude/pre-tool-use-capture.jsonl")
POST_CAPTURE = Path("/tmp/claude/post-tool-use-capture.jsonl")

def clear_captures():
    """Remove prior capture files."""
    PRE_CAPTURE.unlink(missing_ok=True)
    POST_CAPTURE.unlink(missing_ok=True)

def load_events(path: Path) -> list[dict]:
    """Load JSONL events from a capture file."""
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line]

def events_for_tool(events: list[dict], tool_name: str) -> list[dict]:
    """Filter events by tool_name."""
    return [e for e in events if e.get("tool_name") == tool_name]

print(f"Plugin dir: {PLUGIN_DIR}")
print(f"Pre-capture: {PRE_CAPTURE}")
print(f"Post-capture: {POST_CAPTURE}")
print("Ready.")

## Experiment 1: Capture EnterPlanMode hook data

We run `claude -p` with a prompt that triggers plan mode entry. The full-capture plugin records both the PreToolUse event (before EnterPlanMode runs) and the PostToolUse event (after it completes).

**What we're looking for:**
- Does `tool_input` contain the plan file path?
- Does `tool_response` contain the plan file path?
- What other fields are present?

In [ ]:
# Trigger EnterPlanMode — ask the agent to plan something
clear_captures()

PLAN_PROMPT = """Plan how to create a Python script called hello.py that prints hello world.
Enter plan mode, write a brief plan, then exit plan mode."""

cr = run_claude(PLAN_PROMPT, plugin_dir=PLUGIN_DIR, permission_mode="plan")

print("Session ID:", cr.session_id)
print("Exit code:", cr.returncode)
if cr.stderr:
    print("stderr (first 500 chars):", cr.stderr[:500])

# Load captured events
pre_events = load_events(PRE_CAPTURE)
post_events = load_events(POST_CAPTURE)
print(f"\nCaptured {len(pre_events)} PreToolUse event(s)")
print(f"Captured {len(post_events)} PostToolUse event(s)")

# List tools that fired
pre_tools = [e.get("tool_name") for e in pre_events]
post_tools = [e.get("tool_name") for e in post_events]
print(f"\nPreToolUse tools: {pre_tools}")
print(f"PostToolUse tools: {post_tools}")

In [ ]:
# Render EnterPlanMode PreToolUse event (what the hook receives BEFORE the tool runs)
from rich.pretty import pprint

enter_pre = events_for_tool(pre_events, "EnterPlanMode")
print(f"EnterPlanMode PreToolUse events: {len(enter_pre)}")

for i, event in enumerate(enter_pre):
    print(f"\n{'='*60}")
    print(f"  EnterPlanMode PreToolUse #{i+1}")
    print(f"{'='*60}")
    print("\nFull envelope keys:", list(event.keys()))
    print("\nsession_id:", event.get("session_id", "(not present)"))
    print("\ntool_input:")
    pprint(event.get("tool_input", {}))
    # PreToolUse may not have tool_response
    if "tool_response" in event:
        print("\ntool_response:")
        pprint(event["tool_response"])

In [ ]:
# Render EnterPlanMode PostToolUse event (what the hook receives AFTER the tool runs)
enter_post = events_for_tool(post_events, "EnterPlanMode")
print(f"EnterPlanMode PostToolUse events: {len(enter_post)}")

for i, event in enumerate(enter_post):
    print(f"\n{'='*60}")
    print(f"  EnterPlanMode PostToolUse #{i+1}")
    print(f"{'='*60}")
    print("\nFull envelope keys:", list(event.keys()))
    print("\nsession_id:", event.get("session_id", "(not present)"))
    print("\ntool_input:")
    pprint(event.get("tool_input", {}))
    print("\ntool_response:")
    resp = event.get("tool_response", {})
    resp_str = json.dumps(resp) if not isinstance(resp, str) else resp
    if len(resp_str) > 800:
        print(resp_str[:800] + " ... (truncated)")
    else:
        pprint(resp)

# Key question: is the plan file path in tool_response?
if enter_post:
    resp_text = json.dumps(enter_post[0].get("tool_response", {}))
    has_plan_path = "plans/" in resp_text or ".md" in resp_text
    print(f"\n>>> Plan file path found in tool_response: {has_plan_path}")

## Experiment 2: Capture ExitPlanMode hook data

The previous experiment may have also captured ExitPlanMode events if the agent completed the full plan mode cycle. If not, we examine what ExitPlanMode data looks like.

**What we're looking for:**
- Does `tool_input` contain `allowedPrompts`, `pushToRemote`, or other parameters?
- Is `session_id` present? (Can we use it to locate the plan file?)
- Is there any reference to the plan file path?

In [ ]:
# Render ExitPlanMode events from the same capture run
exit_pre = events_for_tool(pre_events, "ExitPlanMode")
exit_post = events_for_tool(post_events, "ExitPlanMode")

print(f"ExitPlanMode PreToolUse events: {len(exit_pre)}")
print(f"ExitPlanMode PostToolUse events: {len(exit_post)}")

for label, events_list in [("PreToolUse", exit_pre), ("PostToolUse", exit_post)]:
    for i, event in enumerate(events_list):
        print(f"\n{'='*60}")
        print(f"  ExitPlanMode {label} #{i+1}")
        print(f"{'='*60}")
        print("\nFull envelope keys:", list(event.keys()))
        print("\nsession_id:", event.get("session_id", "(not present)"))
        print("\ntool_input:")
        pprint(event.get("tool_input", {}))
        if "tool_response" in event:
            print("\ntool_response:")
            resp = event.get("tool_response", {})
            resp_str = json.dumps(resp) if not isinstance(resp, str) else resp
            if len(resp_str) > 800:
                print(resp_str[:800] + " ... (truncated)")
            else:
                pprint(resp)

In [ ]:
# If ExitPlanMode didn't fire, run a second experiment with a more direct prompt
if not exit_pre and not exit_post:
    print("ExitPlanMode did not fire in the first run.")
    print("Trying a more directive prompt...\n")
    
    clear_captures()
    
    DIRECT_PROMPT = """You must enter plan mode using EnterPlanMode, write a one-line plan
to the plan file, then immediately call ExitPlanMode. Do nothing else."""
    
    cr2 = run_claude(DIRECT_PROMPT, plugin_dir=PLUGIN_DIR, permission_mode="plan")
    print("Exit code:", cr2.returncode)
    
    pre_events_2 = load_events(PRE_CAPTURE)
    post_events_2 = load_events(POST_CAPTURE)
    
    exit_pre_2 = events_for_tool(pre_events_2, "ExitPlanMode")
    exit_post_2 = events_for_tool(post_events_2, "ExitPlanMode")
    
    print(f"ExitPlanMode PreToolUse events: {len(exit_pre_2)}")
    print(f"ExitPlanMode PostToolUse events: {len(exit_post_2)}")
    
    for label, events_list in [("PreToolUse", exit_pre_2), ("PostToolUse", exit_post_2)]:
        for i, event in enumerate(events_list):
            print(f"\n{'='*60}")
            print(f"  ExitPlanMode {label} #{i+1}")
            print(f"{'='*60}")
            pprint(event)
else:
    print("ExitPlanMode events captured in the first run — see above.")

## Experiment 3: Plan file location strategies

Regardless of what the hook data contains, we need a strategy to locate the plan file. The proposal identified three options:

| Strategy | Approach | Risk |
|----------|----------|------|
| Most recent in `~/.claude/plans/` | `max(glob, key=mtime)` | Breaks with concurrent sessions |
| Breadcrumb from PostToolUse | PostToolUse writes path to temp file | Requires inter-hook coordination |
| Session context from hook input | Use `session_id` to derive path | Depends on path convention |

Let's test what we can.

In [ ]:
# Strategy A: Most recently modified file in ~/.claude/plans/
import os

plans_dir = Path.home() / ".claude" / "plans"
print(f"Plans directory: {plans_dir}")
print(f"Exists: {plans_dir.exists()}")

if plans_dir.exists():
    plan_files = sorted(plans_dir.glob("*.md"), key=os.path.getmtime, reverse=True)
    print(f"Plan files found: {len(plan_files)}")
    
    for f in plan_files[:5]:
        mtime = os.path.getmtime(f)
        print(f"  {f.name}  (mtime: {mtime})")
    
    if plan_files:
        newest = plan_files[0]
        print(f"\nNewest plan file: {newest}")
        print(f"First 200 chars: {newest.read_text()[:200]}")
else:
    print("Plans directory does not exist — plan mode may store files elsewhere.")

In [ ]:
# Strategy B: Use session_id to derive plan file path
# If plan files follow a convention like session_id.md, we can construct the path

session_id = cr.session_id
print(f"Session ID used: {session_id}")

# Check if any plan file contains or matches the session ID
if plans_dir.exists():
    for f in plans_dir.glob("*.md"):
        if session_id in f.name or session_id in f.stem:
            print(f"MATCH: {f.name} contains session_id")
            break
        # Also check file contents
        content = f.read_text()
        if session_id in content:
            print(f"MATCH: {f.name} contains session_id in content")
            break
    else:
        print("No plan file matches session_id by name or content.")

# Check if session_id was in any hook event
all_events = pre_events + post_events
sessions_seen = set(e.get("session_id", "") for e in all_events)
print(f"Session IDs in hook events: {sessions_seen}")

In [ ]:
# Strategy C: Look for plan file path in the tool_response data
# Search all captured PostToolUse events for file paths

print("Searching all PostToolUse events for file path references...\n")

for event in post_events:
    tool = event.get("tool_name", "unknown")
    resp_str = json.dumps(event.get("tool_response", {}))
    input_str = json.dumps(event.get("tool_input", {}))
    
    # Look for plan file indicators
    for needle in ["plans/", "plan_", ".claude/plans", "plan-file", "planFile"]:
        if needle in resp_str:
            print(f"  [{tool}] tool_response contains '{needle}'")
            print(f"    Context: ...{resp_str[max(0,resp_str.index(needle)-50):resp_str.index(needle)+100]}...")
        if needle in input_str:
            print(f"  [{tool}] tool_input contains '{needle}'")
            print(f"    Context: ...{input_str[max(0,input_str.index(needle)-50):input_str.index(needle)+100]}...")

# Also check full envelope for any top-level keys we might have missed
if post_events:
    print(f"\nAll top-level keys across all PostToolUse events:")
    all_keys = set()
    for event in post_events:
        all_keys.update(event.keys())
    print(f"  {sorted(all_keys)}")

## Findings

Fill in after running the experiments above.

### Q1: What data does PostToolUse receive from EnterPlanMode?

| Field | Value | Notes |
|-------|-------|-------|
| `session_id` | | |
| `tool_name` | `EnterPlanMode` | Expected |
| `tool_input` | | |
| `tool_response` | | Key question: does it contain the plan file path? |

### Q2: What data does PreToolUse receive for ExitPlanMode?

| Field | Value | Notes |
|-------|-------|-------|
| `session_id` | | |
| `tool_name` | `ExitPlanMode` | Expected |
| `tool_input` | | `allowedPrompts`? `pushToRemote`? |

### Q6: How does the agent hook locate the plan file?

| Strategy | Feasibility | Notes |
|----------|-------------|-------|
| Most recent in `~/.claude/plans/` | | |
| Session ID correlation | | |
| Path in tool_response | | |
| Breadcrumb temp file | Always works | Fallback — PostToolUse writes path for PreToolUse to read |